In [ ]:
import numpy as np
from matplotlib import pyplot as plt
import scipy.io

In [ ]:
from inference.ploting_routines import plot_real_vs_synthetic,plot_real_and_estimated

from models.froehlich_model_small import FroehlichModelSmall
model_small = FroehlichModelSmall(network_idx=2)

from models.froehlich_model_large import FroehlichModelLarge
model_large = FroehlichModelLarge(network_idx=2)

obs_data, true_pop_parameters = model_small.load_data(n_data=10000, load_eGFP=True, load_d2eGFP=True)

In [ ]:
mat_small = scipy.io.loadmat('data/code/S=[5,6]_transfection_norm.mat') #scipy.io.loadmat('data/code/results_norm.mat')
mat_large = scipy.io.loadmat('data/code/S=[5,6]_transfection_enzdegribo.mat') #scipy.io.loadmat('data/code/results_enzdegribo.mat')

In [ ]:
names_small = mat_small['parameters_MEM'][0,0][3].flatten()
names_large = mat_large['parameters_MEM'][0,0][3].flatten()
names_small, names_large

In [ ]:
param_estimates_small = mat_small['parameters_MEM'][0,0][6][0][0][4] #1
param_estimates_large = mat_large['parameters_MEM'][0,0][6][0][0][4] #1
param_estimates_small

In [ ]:
likelihood_small = mat_small['parameters_MEM'][0,0][6][0][0][2]
likelihood_large = mat_large['parameters_MEM'][0,0][6][0][0][2]
best_id_small = np.nanargmax(likelihood_small)
best_id_large = np.nanargmax(likelihood_large)
likelihood_small[best_id_small], likelihood_large[best_id_large]

In [ ]:
mean_small = param_estimates_small[:model_small.n_params, best_id_small]
var_small = np.exp(param_estimates_small[model_small.n_params:, best_id_small])
cov_small = np.diag(var_small)

mean_large = param_estimates_large[:model_large.n_params, best_id_large]
var_large = np.exp(param_estimates_large[model_large.n_params:, best_id_large])
cov_large = np.diag(var_large)

In [ ]:
mean_small, var_small, mean_large, var_large

In [ ]:
x_limits = [(1e-2, 10), (1e-3, 1), (1e-2, 1), (1e2, 1e3),
            (np.power(10, -0.5), np.power(10, 0.5)), (1, np.power(10, 1.5))]
for i_n, name in enumerate(names_small[:6]):
    plt.figure()
    plt.title(name[0][6:-1])
    xmax = mean_small[i_n] + 4*np.sqrt(cov_small.diagonal()[i_n])
    xmin = mean_small[i_n] - 4*np.sqrt(cov_small.diagonal()[i_n])
    xs = np.linspace(xmin, xmax, 100)
    gaussian = scipy.stats.norm(mean_small[i_n], np.sqrt(cov_small.diagonal()[i_n]))
    plt.plot(np.power(10, xs), gaussian.pdf(xs), color='red', label='(i)')

    #samples = np.random.normal(mean_small[i_n], np.sqrt(cov_small.diagonal()[i_n]), 10000)
    #samples = np.power(10, samples)
    #samples_log = np.log(samples)
    #res = stats.fit(stats.norm, data=samples_log)
    #gaussian = scipy.stats.norm(res.params.loc, res.params.scale)
    #xmax = res.params.loc + 4*res.params.scale
    #xmin = res.params.loc - 4*res.params.scale
    #xs = np.linspace(xmin, xmax, 100)
    #plt.plot(np.exp(xs), gaussian.pdf(xs), color='red', marker='x', label='(i)')

    if i_n not in [0, 3]:
        xmax = mean_large[i_n] + 4*np.sqrt(cov_large.diagonal()[i_n])
        xmin = mean_large[i_n] - 4*np.sqrt(cov_large.diagonal()[i_n])
        xs = np.linspace(xmin, xmax, 100)
        gaussian = scipy.stats.norm(mean_large[i_n], np.sqrt(cov_large.diagonal()[i_n]))
        plt.plot(np.power(10, xs), gaussian.pdf(xs), color='purple', label='(iv)')

    plt.xscale('log', base=10)
    plt.xlim(x_limits[i_n])
    plt.show()

In [ ]:
simulator_small =  model_small.build_simulator(with_noise=False, exp_func='power10')
simulator_large =  model_large.build_simulator(with_noise=False, exp_func='power10')

In [ ]:
model_name = 'FroehlichModelLarge'

if model_name == 'FroehlichModelSmall':
    mean_gfp = mean_small[[0,1,3,4,5,5]]
    cov_gfp = np.diag(cov_small.diagonal()[[0,1,3,4,5,5]])
    mean_gfpd2 = mean_small[[0,2,3,4,5,5]]
    cov_gfpd2 = np.diag(cov_small.diagonal()[[0,2,3,4,5,5]])
    simulator = simulator_small
else:
    mean_gfp = mean_large[[0,10,9,3,8,6,7,1,4,5,5]]
    cov_gfp = np.diag(cov_large.diagonal()[[0,10,9,3,8,6,7,1,4,5,5]])
    mean_gfpd2 = mean_large[[0,10,9,3,8,6,7,2,4,5,5]]
    cov_gfpd2 = np.diag(cov_large.diagonal()[[0,10,9,3,8,6,7,2,4,5,5]])
    simulator = simulator_large

In [ ]:
names_large[[0,10,9,3,8,6,7,1,4,5,5]]

In [ ]:
model_large.param_names

In [ ]:
plot_real_vs_synthetic(estimated_mean=mean_gfp,
                           estimated_cov=cov_gfp,
                           data=obs_data[0],
                           model_name=model_name,
                           n_trajectories=len(obs_data[0])*10,
                           simulator=simulator,
                           save_fig=model_name+'_eGFP_dif_fröhlich',
                           estimation_function=np.median,
                           ylim=(-1.,1.),
                           seed=0)
plot_real_and_estimated(estimated_mean=mean_gfp,
                           estimated_cov=cov_gfp,
                           data=obs_data[0],
                           model_name=model_name,
                           n_trajectories=len(obs_data[0]),
                           simulator=simulator,
                           save_fig=model_name+'_eGFP_estimate_fröhlich',
                           seed=0)

In [ ]:
plot_real_vs_synthetic(estimated_mean=mean_gfpd2,
                           estimated_cov=cov_gfpd2,
                           data=obs_data[1],
                           model_name=model_name,
                           n_trajectories=len(obs_data[1])*10,
                           simulator=simulator,
                           save_fig=model_name+'_eGFPd2_dif_fröhlich',
                           #estimation_function=np.median,
                           ylim=(-1.,1.),
                           seed=0)
plot_real_and_estimated(estimated_mean=mean_gfpd2,
                           estimated_cov=cov_gfpd2,
                           data=obs_data[1],
                           model_name=model_name,
                           n_trajectories=len(obs_data[1]),
                           simulator=simulator,
                           save_fig=model_name+'_eGFPd2_estimate_fröhlich',
                           seed=0)